# Prediction with linear regression

In [28]:
import numpy as np
import pandas as pd
from pandas.core.interchange.from_dataframe import primitive_column_to_ndarray
##from pandas import read_csv as rc
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


In [29]:
df = pd.read_csv('data_cleaned.csv')

In [30]:
df['city'] = df['city'].astype('category')
df['district'] = df['district'].astype('category')
df['neighborhood'] = df['neighborhood'].astype('category')
df['room'] = df['room'].astype('int')
df['living_room'] = df['living_room'].astype('int')
df['area_mSqr'] = df['area_mSqr'].astype('int')
df['building_age'] = df['building_age'].astype('int')
df['floor'] = df['floor'].astype('float')
df['list-view-price'] = df['list-view-price'].astype('float')
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1387 entries, 0 to 1386
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   city             1387 non-null   category
 1   district         1387 non-null   category
 2   neighborhood     1387 non-null   category
 3   room             1387 non-null   int64   
 4   living_room      1387 non-null   int64   
 5   area_mSqr        1387 non-null   int64   
 6   building_age     1387 non-null   int64   
 7   floor            1387 non-null   float64 
 8   list-view-price  1387 non-null   float64 
dtypes: category(3), float64(2), int64(4)
memory usage: 73.2 KB
None


In [31]:
categorical_features = ['city', 'district', 'neighborhood']
numerical_features = ['room', 'living_room', 'area_mSqr', 'building_age', 'floor']

In [32]:
full_pipeline = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

In [33]:
X = df.drop('list-view-price' , axis=1)
y= df['list-view-price']


In [34]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)
##random_state = bir daha ayni seti yakaalmak istersek kullancagimiz random number generator daki seed , fix lememk istersek kullaniriz

In [35]:
model = Pipeline([
    ('preprocessor', full_pipeline),
    ('model', LinearRegression())
])

In [36]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [37]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'R2: {r2}')

MSE: 33733104.67414118
RMSE: 5808.020719155639
R2: 0.5532215434934975


In [38]:
feature_importances = model.named_steps['model'].coef_
print(len(feature_importances))
print(feature_importances)

327
[ 2.29118267e+02  0.00000000e+00  3.55051539e+03 -3.13054385e+03
  1.04606250e+02 -5.50895294e+03  5.50895294e+03 -2.57275302e+03
 -4.66427112e+03  2.54446313e+03  8.28738925e+03 -3.60364212e+03
 -9.11629293e+03  1.10206076e+03 -4.86083128e+03 -2.62686492e+03
 -1.23665758e+03  2.45234126e+03  9.73633736e+03  1.98061084e+04
  9.94632150e+02 -2.71679222e+03  7.25299532e+03 -5.03683782e+03
 -3.99124762e+03 -1.82993532e+03 -9.18335587e+03 -6.91627346e+03
  9.43888626e+03  5.96419917e+02 -2.95627023e+03  4.60144370e+02
  9.63620624e+02 -2.53774284e+03 -7.86586245e+03 -7.95086931e+03
  1.79439247e+03  6.83073598e+03  3.05475832e+02  4.02950094e+03
 -3.58945370e+03 -2.32214652e+03  8.98259629e+03  1.66995145e+01
  2.59217869e+03  4.58106878e+03  2.82796124e+03 -2.50135790e+03
  1.13166499e+04 -5.18903124e+03  7.42600802e+03  5.53163467e+03
  4.07419519e+03 -1.92291071e+03  7.74586712e+03 -2.32380443e+03
 -2.99317225e+03 -1.52285747e+04  6.63715394e+02  1.81859118e+03
 -5.55254212e+03 -5.5

In [39]:
print("Categorical Features")
for i in range(len(categorical_features)):
    for j in range(len(model.named_steps['preprocessor'].transformers_[1][1].categories_[i])):
        print(model.named_steps['preprocessor'].transformers_[1][1].categories_[i][j], feature_importances[len(numerical_features)+j])

Categorical Features
Manisa  -5508.952943316802
İzmir  5508.95294331004
 Akhisar  -5508.952943316802
 Alaşehir  5508.95294331004
 Aliağa  -2572.753020764485
 Balçova  -4664.271115938799
 Bayraklı  2544.4631299349003
 Bergama  8287.389250856555
 Bornova  -3603.6421200632108
 Buca  -9116.292931614767
 Demirci  1102.0607646670437
 Dikili  -4860.83128419387
 Foça  -2626.86492285309
 Gaziemir  -1236.6575786162166
 Güzelbahçe  2452.3412615118946
 Karabağlar  9736.33735910815
 Karaburun  19806.10842346565
 Karşıyaka  994.63214996789
 Kemalpaşa  -2716.792221483819
 Konak  7252.995315977121
 Kırkağaç  -5036.837821503932
 Menderes  -3991.247622312556
 Menemen  -1829.9353190188717
 Narlıdere  -9183.355868748478
 Salihli  -6916.273459735959
 Saruhanlı  9438.886260660895
 Seferihisar  596.4199173108897
 Selçuk  -2956.2702343299916
 Soma  460.14437037224116
 Tire  963.6206236387636
 Torbalı  -2537.742841929912
 Turgutlu  -7865.862452656072
 Urla  -7950.8693132188155
 Yunusemre  1794.3924744262633
 Ç

In [40]:

new_data = pd.DataFrame({
    'city': ['Manisa'+' '],
    'district' : [' '+'Yunusemre'+' '],
    'neighborhood' : [' '+'Muradiye Mah.'],
    'room' : [2],
    'living_room' : [1],
    'area_mSqr' : [90],
    'building_age' : [5],
    'floor' : [3]
})
model.predict(new_data)

array([20010.88069147])

In [41]:
print(df[(df['city'] == 'Manisa ') & (df['district'] == ' Yunusemre ')])

        city     district    neighborhood  room  living_room  area_mSqr  \
223  Manisa    Yunusemre    Muradiye Mah.     2            1         85   
228  Manisa    Yunusemre    Muradiye Mah.     1            1         45   
230  Manisa    Yunusemre    Muradiye Mah.     1            1         49   
232  Manisa    Yunusemre       Mutlu Mah.     2            1        105   
235  Manisa    Yunusemre     75. Yıl Mah.     3            1        130   
..       ...          ...             ...   ...          ...        ...   
640  Manisa    Yunusemre    Muradiye Mah.     2            1         85   
643  Manisa    Yunusemre    Muradiye Mah.     2            1         98   
644  Manisa    Yunusemre    Muradiye Mah.     1            1         50   
645  Manisa    Yunusemre    Muradiye Mah.     2            1         95   
655  Manisa    Yunusemre    Muradiye Mah.     2            1         90   

     building_age  floor  list-view-price  
223             8    3.0          18000.0  
228        